BUỔI 1.2 (DELTA LAKE)

Bước 0 — Kiểm tra dữ liệu

In [0]:
dbutils.fs.ls("/Volumes/workspace/lab_tuan1/raw/")


Bước 1 — Đọc file mà chưa tạo bảng

In [0]:
path = "/Volumes/workspace/lab_tuan1/raw/orders_batch1.csv"

df_raw = spark.read.csv(path, header=True, inferSchema=False)   # đọc hết thành STRING
df_raw.printSchema()
df_raw.show(5, truncate=False)


Bước 2 — Tạo bảng Delta với schema tường minh

2.1 Tạo bảng rỗng

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_tuan1.bronze_orders_tuan1 (
    order_id    STRING,
    shop_code   STRING,
    order_ts    TIMESTAMP,      -- đã xử lý 2 định dạng ở bước nạp
    amount      DECIMAL(18,2),  -- KHÔNG dùng DOUBLE cho tiền
    quantity    INT,
    product_id  STRING,
    status      STRING,
    channel     STRING,
    updated_at  TIMESTAMP
);

2.2 Nạp dữ liệu

In [0]:
%sql
INSERT INTO workspace.lab_tuan1.bronze_orders_tuan1 
    SELECT 
        order_id, 
        shop_code,
        coalesce(
            try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss'),
            try_to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')
        ) AS order_ts,
        cast(amount AS DECIMAL(18,2)),
        cast(quantity AS INT),
        product_id,
        status,
        channel,
        try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')
    FROM read_files(
        '/Volumes/workspace/lab_tuan1/raw/orders_batch1.csv',
        format => 'csv', header => true
    )

In [0]:
%sql
select * from workspace.lab_tuan1.bronze_orders_tuan1

In [0]:
%sql
SELECT count(*) AS SO_DONG , count(DISTINCT order_id) AS SO_DON_HANG

FROM workspace.lab_tuan1.bronze_orders_tuan1 

In [0]:
%sql
select count(*) from workspace.lab_tuan1.bronze_orders_tuan1
where order_ts is null

In [0]:
%sql
-- (a) Tổng THÔ, không dedupe

select sum(amount) as DoanhThu_tho
from workspace.lab_tuan1.bronze_orders_tuan1
where upper(trim(status)) = 'PAID'

In [0]:
%sql
-- (b) Tổng SAU KHI dedupe trong truy vấn: mỗi order_id giữ dòng updated_at lớn nhất (Dùng row_number để dánh thứ tự 1, 2 , 3,... 
                                                                                        --còn rank và deset rank sẽ có cùng cấp nên --không dùng ) -> mục tiêu không trùng

select sum(amount) as doanhthu_dedup
from (
    select 
        amount, 
        status,
        row_number() over(partition by order_id order by updated_at desc) AS rn
    from workspace.lab_tuan1.bronze_orders_tuan1
)
where rn = 1 and upper(trim(status)) = 'PAID'

Bước 3 — UPDATE có kiểm soát

3.1 Xem trước và ghi lại giá trị cũ


In [0]:
%sql
SELECT order_id, order_ts, status, amount
from workspace.lab_tuan1.bronze_orders_tuan1
WHERE order_id = 'O1027'

3.2 Chạy UPDATE


In [0]:
%sql
update workspace.lab_tuan1.bronze_orders_tuan1
set order_ts = timestamp '2026-10-03 13:15:00'
where order_id = 'O1027'

3.3 Kiểm chứng


In [0]:
%sql
SELECT order_id, order_ts, status, amount
from workspace.lab_tuan1.bronze_orders_tuan1
WHERE order_id = 'O1027'

In [0]:
%sql
DESCRIBE  HISTORY workspace.lab_tuan1.bronze_orders_tuan1

Bước 4 — MERGE upsert

4.1 Chuẩn bị nguồn

In [0]:
%sql
SELECT order_id, COUNT(*) AS so_lan
FROM read_files('/Volumes/workspace/lab_tuan1/raw/orders_increment.csv',
                format => 'csv', header => true)
GROUP BY order_id
HAVING COUNT(*) > 1;


4.2 Tạo bảng staging(tạm) đã dedupe




In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_increment AS
SELECT * FROM (
  SELECT
    order_id, shop_code,
    coalesce(to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss'),
             to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')) AS order_ts,
    cast(amount AS DECIMAL(18,2)) AS amount,
    cast(quantity AS INT) AS quantity,
    product_id, status, channel,
    to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss') AS updated_at,
    row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
  FROM read_files('/Volumes/workspace/lab_tuan1/raw/orders_increment.csv',
                  format => 'csv', header => true)
) WHERE rn = 1;

SELECT COUNT(*) FROM stg_increment;


4.3 MERGE

In [0]:
%sql
MERGE INTO workspace.lab_tuan1.bronze_orders_tuan1 AS t
USING stg_increment AS s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET
  t.shop_code = s.shop_code,
  t.order_ts  = s.order_ts,
  t.amount    = s.amount,
  t.quantity  = s.quantity,
  t.status    = s.status,
  t.channel   = s.channel,
  t.updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT *;


In [0]:
%sql
SELECT COUNT(*) AS so_dong, COUNT(DISTINCT order_id) AS so_order
FROM workspace.lab_tuan1.bronze_orders_tuan1;


In [0]:
%sql
-- revenue: nhớ dedupe, đừng SUM thô
SELECT SUM(amount) AS revenue_paid
FROM (
  SELECT amount, status,
         row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
  FROM workspace.lab_tuan1.bronze_orders_tuan1
) WHERE rn = 1 AND upper(trim(status)) = 'PAID';


Bước 5 — Time travel

In [0]:
%sql
-- Trạng thái ban đầu (version 0): 40 dòng
SELECT COUNT(*) AS so_dong
FROM workspace.lab_tuan1.bronze_orders_tuan1 VERSION AS OF 0;


In [0]:
%sql
-- Tổng tại version 0, KHÔNG dedupe (vì bảng gốc còn 2 cặp trùng)
SELECT SUM(amount) AS revenue_tho_v0
FROM workspace.lab_tuan1.bronze_orders_tuan1 VERSION AS OF 0
WHERE upper(trim(status)) = 'PAID';


In [0]:
%sql
-- Trạng thái hiện tại: 45 dòng
SELECT COUNT(*) AS so_dong, COUNT(DISTINCT order_id) AS so_order
FROM workspace.lab_tuan1.bronze_orders_tuan1;


Bước 6 — Small files & OPTIMIZE

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_tuan1.opt_demo_tuan1 (
  id INT, note STRING
);


In [0]:
for i in range(20):
    spark.sql(f"INSERT INTO workspace.lab_tuan1.opt_demo_tuan1 VALUES ({i}, 'row {i}')")
print("xong 20 lan ghi")


In [0]:
%sql
DESCRIBE DETAIL workspace.lab_tuan1.opt_demo_tuan1;


In [0]:
%sql
OPTIMIZE workspace.lab_tuan1.opt_demo_tuan1;


Bước 7 — VACUUM và chứng minh time travel mất

7.1 Tạo version để có cái mà mất


In [0]:
%sql
INSERT INTO workspace.lab_tuan1.opt_demo_tuan1 VALUES (999, 'before vacuum');


7.2 Chạy VACUUM

In [0]:
# Chỉ dùng cho bảng demo. KHÔNG chạy trên bảng thật.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")



In [0]:
%sql
VACUUM workspace.lab_tuan1.opt_demo_tuan1 RETAIN 168 HOURS;

In [0]:
%sql
DESCRIBE HISTORY workspace.lab_tuan1.opt_demo_tuan1;

7.3 Kiểm chứng

In [0]:
%sql
SELECT * FROM workspace.lab_tuan1.opt_demo_tuan1 VERSION AS OF 23; --lấy version mới nhất trừ 1. VD ở trên 24-1=23


Bước 8 — Liquid Clustering

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_tuan1.fact_demo_tuan1
CLUSTER BY (shop_code, order_ts)
AS
SELECT order_id, shop_code, order_ts, amount
FROM workspace.lab_tuan1.bronze_orders_tuan1


In [0]:
%sql
DESCRIBE DETAIL workspace.lab_tuan1.fact_demo_tuan1;


In [0]:
%sql
ALTER TABLE workspace.lab_tuan1.fact_demo_tuan1 CLUSTER BY (order_ts);


Bước 9 — Doanh thu theo ngày

In [0]:
%sql
SELECT
  CAST(order_ts AS DATE) AS ngay,
  COUNT(*) AS so_don,
  SUM(amount) AS doanh_thu
FROM (
  SELECT order_id, order_ts, amount, status, updated_at,
         row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
  FROM workspace.lab_tuan1.bronze_orders_tuan1
)
WHERE rn = 1
  AND upper(trim(status)) = 'PAID'
  AND amount IS NOT NULL        -- đơn thiếu số tiền không xếp được vào doanh thu
GROUP BY CAST(order_ts AS DATE)
ORDER BY ngay;
